# NAIVE RAG Pipeline
### PDF → Chunking → FAISS → Query → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")
print(f"Sample Chunk:\n{chunks[0].page_content}")

Total Chunks: 136
Sample Chunk:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director General of Defence Research and
Development Organisation
In office
1992–1999
A. P. J. Abdul Kalam
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl
kəˈlɑːm/ ⓘ UB-duul k ə -LAHM; 15 October 1931 –
27 July 2015) was an Indian aerospace scientist and
statesman who served as the president of India from
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a


## Step 3: Create Embeddings & Store in FAISS

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Create Retriever

In [5]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

## Step 5: Setup LLM & Query

In [6]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

query = "Where was Abdul Kalam born?"

# Retrieve relevant chunks
retrieved_docs = retriever.invoke(query)

In [13]:
type(retrieved_docs)

list

In [12]:
print(retrieved_docs[0])

page_content='Organisation
Website A. P. J. Abdul Kalam Centre (h
ttps://kalamcentre.com)
Signature
Kalam's birthplace in Rameswaram,
Tamil Nadu
Avul Pakir Jainulabdeen Abdul Kalam was born on 15
October 1931 to a Tamil Muslim family in the
pilgrimage center of Rameswaram on Pamban Island,
Madras Presidency (now in the Indian state of Tamil
Nadu).[2][3] His father, Jainulabdeen Marakayar, was a
boat owner and imam of a local mosque,[4] and his
mother, Ashiamma, was a housewife.[5][6] His father
owned a boat that ferried Hindu pilgrims between
Rameswaram and Dhanushkodi.[7][8]
Kalam was the youngest of four brothers and a sister in
the family.[9][10][11] His ancestors had been wealthy
Marakayar traders and landowners, with numerous
properties and large tracts of land. Marakayar are a
Muslim ethnic group found in coastal Tamil Nadu and
Sri Lanka who claim descent from Arab traders and
local women. The family business had involved trading
goods and transporting passengers between the Indi

In [14]:
# Build context from retrieved chunks
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Create prompt
prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

# Get response
response = llm.invoke(prompt)
print("Answer:", response.content)

Answer: Abdul Kalam was born in Rameswaram, Tamil Nadu, on Pamban Island.


## Retrieved Source Chunks

In [15]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:300])


--- Chunk 1 (Page 2) ---
Organisation
Website A. P. J. Abdul Kalam Centre (h
ttps://kalamcentre.com)
Signature
Kalam's birthplace in Rameswaram,
Tamil Nadu
Avul Pakir Jainulabdeen Abdul Kalam was born on 15
October 1931 to a Tamil Muslim family in the
pilgrimage center of Rameswaram on Pamban Island,
Madras Presidency (now 

--- Chunk 2 (Page 10) ---
of Tamil Nadu announced that Kalam's birthday, 15 October, would be observed as "Youth Renaissance
Day". It also instituted the "Dr. A. P. J. Abdul Kalam Award" constituting a gold medal, a certificate and
₹500,000 (US$5,900), to be awarded annually on the Indian Independence Day, to residents of th

--- Chunk 3 (Page 11) ---
A P J Abdul Kalam: The Visionary of India by K Bhushan, G Katyal; A P H Pub Corp,
2002.[176]
The Kalam Effect: My Years with the President by P M Nair; HarperCollins, 2008.[177]
My Days With Mahatma Abdul Kalam by Fr A K George; Novel Corporation, 2009.[178]
A.P.J. Abdul Kalam: A Life by Arun Tiwari


Test Questions

1. What year was Abdul Kalam born?
2. What was the name of the coronary stent Kalam co-developed?
3. Which institution did Kalam study aerospace engineering at?
4. How many electoral votes did Kalam receive in the 2002 presidential election?
5. What was Kalam's role at DRDO from 1992 to 1999?